<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/8-reranker-instructions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Reranker Instructions with Qwen3

In this notebook we demonstrate how to use **reranker instructions** with Vectara's `qwen3-reranker`. Reranker instructions let you pass domain-specific guidance to the reranker so it can better score relevance for your particular use case.

We'll cover:
- Baseline query using qwen3-reranker **without** instructions
- **Role-based intent steering**: using instructions to shift results toward practical docs vs academic papers
- **Abbreviation and jargon resolution**: using a glossary to help the reranker understand domain-specific terms

## About Vectara

[Vectara](https://vectara.com/) is the Agent Platform for trusted enterprise AI: a unified Agentic RAG platform with built-in multi-modal retrieval, orchestration, and always-on governance. Deploy it on-prem (air-gapped), in your VPC, or as SaaS. Vectara agents deliver grounded answers and safe actions with source citations, step-level audit trails, fine-grained access controls, and real-time policy and factual-consistency enforcement, so teams ship faster with lower risk, and with trusted, production-grade AI agents at scale.

Vectara provides a complete API-first platform for building production RAG and agentic applications:

- **Simple Integration**: RESTful APIs and SDKs (Python, JavaScript) for quick integration into any stack
- **Flexible Deployment**: Choose SaaS, VPC, or on-premises deployment based on your requirements
- **Multi-Modal Support**: Index and search across text, tables, and images from PDFs, documents, and structured data
- **Advanced Retrieval**: Hybrid search combining semantic and keyword matching with state-of-the-art reranking
- **Grounded Generation**: LLM responses with citations and factual consistency scores to reduce hallucinations
- **Enterprise-Ready**: Built-in access controls, audit logging, and compliance (SOC2, HIPAA) from day one

## Getting Started

This notebook assumes you've completed Notebooks 1 and 2:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs) with Boomerang embeddings
- Notebook 2: Ingested AI research papers and Vectara documentation

In [1]:
import os
import requests
import json

# Set up authentication
api_key = os.environ['VECTARA_API_KEY']

# Corpus keys from Notebook 1
research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

# Base URL for Vectara API v2
BASE_URL = "https://api.vectara.io/v2"

# Common headers for all requests
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'x-api-key': api_key
}


def run_query(query_request):
    """Run a Vectara query and print the summary and top results."""
    response = requests.post(f"{BASE_URL}/query", headers=headers, json=query_request)
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

    result = response.json()
    print("\n=== Generated Summary ===")
    print(result['summary'])
    print(f"\n=== Factual Consistency Score: {result.get('factual_consistency_score', 'N/A')} ===")

    print("\n=== Top Search Results ===")
    for i, sr in enumerate(result.get('search_results', [])[:5], 1):
        meta = sr.get('document_metadata', {})
        print(f"\n--- Result {i} (score: {sr.get('score', 'N/A'):.4f}) ---")
        print(f"Document: {sr.get('document_id', 'N/A')}")
        print(f"Title: {meta.get('title', 'N/A')}")
        print(f"Text: {sr['text'][:200]}...")

    return result


print("Setup complete.")

Setup complete.


## Example 1: Baseline Query — Qwen3 Reranker Without Instructions

First, let's run a query using the `qwen3-reranker` **without** any instructions across **both corpora** (research papers and Vectara docs). This establishes a baseline to compare against in the following examples.

Note the API difference from earlier notebooks: instead of `reranker_id`, we use `reranker_name` to select the qwen3 reranker.

In [2]:
QUERY = "How does reranking improve search result quality?"

baseline_request = {
    "query": QUERY,
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 100
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ]
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

print("=" * 60)
print("BASELINE: qwen3-reranker without instructions")
print("=" * 60)
baseline_result = run_query(baseline_request)

BASELINE: qwen3-reranker without instructions

=== Generated Summary ===
Reranking improves search result quality by refining and reordering search results after their initial retrieval. This process enhances the relevance of the results by using various reranker types, such as neural models, which improve precision, and methods like Maximum Marginal Relevance (MMR) to add diversity and reduce redundancy. Rerankers can be configured to prioritize the most relevant and business-critical results, ensuring they appear at the top according to specific ranking logic. This approach is particularly beneficial in applications requiring advanced neural ranking capabilities, such as multilingual content and real-time response generation, thereby optimizing search result quality for different use cases [1], [5], [6].

=== Factual Consistency Score: 0.79296875 ===

=== Top Search Results ===

--- Result 1 (score: 0.9991) ---
Document: docs-vectara-com-docs-sdk-python-rerankers
Title: Rerankers
Tex

## Example 2: Role-Based Intent Steering

Now we use the **same query** with and without instructions, searching **both corpora**. The two corpus types (academic papers vs Vectara product docs) create a natural contrast.

Without instructions, the reranker returns a mix of results from both corpora. With instructions that describe the user as a developer building a production application, we expect the reranker to prioritize practical Vectara documentation over academic papers.

In [3]:
# First: query WITHOUT instructions (same query as baseline, same corpora)
no_instructions_request = {
    "query": QUERY,
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 100,
                    "cutoff": 0.2
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ]
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

print("=" * 60)
print("WITHOUT INSTRUCTIONS: mixed results from both corpora")
print("=" * 60)
no_instructions_result = run_query(no_instructions_request)

# Now: same query WITH role-based instructions
ROLE_INSTRUCTIONS = (
    "The user is a software developer building a production search application. "
    "Prioritize practical implementation guides, API documentation, and "
    "configuration options over academic research papers and theoretical analysis."
)

with_instructions_request = {
    "query": QUERY,
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "instructions": ROLE_INSTRUCTIONS,
                    "limit": 100,
                    "cutoff": 0.2
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ]
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

print("\n" + "=" * 60)
print("WITH INSTRUCTIONS: prioritize practical docs for developers")
print("=" * 60)
with_instructions_result = run_query(with_instructions_request)

WITHOUT INSTRUCTIONS: mixed results from both corpora

=== Generated Summary ===
Reranking improves search result quality by refining and reordering the results after the initial retrieval. This process enhances the relevance of the results by ensuring that the most pertinent and business-critical results appear at the top. Rerankers can be configured to optimize result quality for different use cases, such as improving precision with neural models, adding diversity, or applying custom business logic. By using advanced neural ranking capabilities, rerankers can enhance relevance scoring and support multilingual content, making them ideal for applications requiring high-quality search results [1], [4], [6].

=== Factual Consistency Score: 0.87109375 ===

=== Top Search Results ===

--- Result 1 (score: 0.9991) ---
Document: docs-vectara-com-docs-sdk-python-rerankers
Title: Rerankers
Text: Rerankers enhance the relevance of search results by refining and reordering them
after initial ret

In [4]:
# Compare: count papers vs docs in top-5 for both variants
def classify_result(doc_id):
    """Classify a result as 'paper' or 'docs' based on document ID."""
    if doc_id.endswith('.pdf'):
        return 'paper'
    return 'docs'

if no_instructions_result and with_instructions_result:
    print("=== Role-Based Intent Steering Comparison ===\n")

    for label, result in [("Without instructions", no_instructions_result),
                          ("With instructions", with_instructions_result)]:
        top5 = result.get('search_results', [])[:5]
        papers = sum(1 for sr in top5 if classify_result(sr['document_id']) == 'paper')
        docs = sum(1 for sr in top5 if classify_result(sr['document_id']) == 'docs')
        print(f"{label} — top-5 breakdown: {papers} papers, {docs} docs")
        for i, sr in enumerate(top5, 1):
            kind = classify_result(sr['document_id'])
            print(f"  {i}. [{kind:5s}] {sr['document_id']} (score: {sr.get('score', 0):.4f})")
        print()

    print("-> With developer-focused instructions, Vectara docs should dominate the top results.")

=== Role-Based Intent Steering Comparison ===

Without instructions — top-5 breakdown: 0 papers, 5 docs
  1. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.9991)
  2. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.9074)
  3. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.9059)
  4. [docs ] docs-vectara-com-docs-rest-api-query-corpus (score: 0.9011)
  5. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.8997)

With instructions — top-5 breakdown: 0 papers, 5 docs
  1. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.9952)
  2. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.9012)
  3. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.8602)
  4. [docs ] docs-vectara-com-docs-console-ui-configure-queries (score: 0.8377)
  5. [docs ] docs-vectara-com-docs-sdk-python-rerankers (score: 0.8349)

-> With developer-focused instructions, Vectara docs should dominate the top results.


## Example 3: Abbreviation and Jargon Resolution

One of the most powerful uses of reranker instructions is **resolving domain-specific abbreviations and jargon**. In specialized domains, queries often use abbreviations that don't appear literally in the documents. Instructions help the reranker bridge this gap.

The key is using a **terse query with minimal context clues** — if the query itself explains the abbreviation (e.g., "How does FCS help detect hallucinations in LLM outputs?"), the reranker can figure it out from surrounding words alone. A short, opaque query like `"HHEM accuracy and benchmarks"` genuinely requires the glossary.

In [5]:
# A terse query using a domain abbreviation with NO context clues.
# "HHEM" alone gives the reranker nothing to work with.
ABBREV_QUERY = "HHEM accuracy and benchmarks"

# Without instructions — the reranker treats "HHEM" as an opaque string
no_glossary_request = {
    "query": ABBREV_QUERY,
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 100,
                    "cutoff": 0.2
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ]
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

print("=" * 60)
print("WITHOUT GLOSSARY: 'HHEM' is opaque to the reranker")
print("=" * 60)
no_glossary_result = run_query(no_glossary_request)

WITHOUT GLOSSARY: 'HHEM' is opaque to the reranker

=== Generated Summary ===
The Hughes Hallucination Evaluation Model (HHEM) is used to assess the factual consistency of AI-generated summaries. It provides a calibrated score ranging from 0.0 to 1.0, where a higher score indicates a higher confidence in the summary's factual consistency. For example, a score of 0.95 suggests a 95% likelihood that the summary is free of hallucinations [1].

In terms of benchmarks, HHEM-2.1 has a balanced accuracy (BA) of 55.27% and an F1-Macro score of 40.30% [5]. Other versions, such as HHEM-2.1-Open, have a BA of 51.98% and an F1-M of 33.03% [5]. These scores indicate moderate performance in detecting hallucinations compared to other models [6].

=== Factual Consistency Score: 0.9453125 ===

=== Top Search Results ===

--- Result 1 (score: 0.9878) ---
Document: docs-vectara-com-docs-hallucination-and-evaluation-hallucination-evaluation
Title: Hallucination evaluation
Text: Vectara uses theHughes Hall

In [6]:
# With glossary instructions — tell the reranker what HHEM means
GLOSSARY_INSTRUCTIONS = (
    "This corpus contains Vectara platform documentation and AI research papers. "
    "HHEM stands for Hughes Hallucination Evaluation Model, Vectara's model for "
    "evaluating factual consistency of LLM-generated text. "
    "Prioritize results about hallucination detection, factual grounding, "
    "and evaluation metrics."
)

with_glossary_request = {
    "query": ABBREV_QUERY,
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "instructions": GLOSSARY_INSTRUCTIONS,
                    "limit": 100,
                    "cutoff": 0.2
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ]
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

print("=" * 60)
print("WITH GLOSSARY: reranker knows HHEM = Hughes Hallucination Evaluation Model")
print("=" * 60)
with_glossary_result = run_query(with_glossary_request)

WITH GLOSSARY: reranker knows HHEM = Hughes Hallucination Evaluation Model

=== Generated Summary ===
The search results provide information on the accuracy and benchmarks of the Hallucination Detector models, specifically the HHEM series. The Balanced Accuracy (BA) and F1-Macro scores are used to evaluate these models. For HHEM-2.1, the BA ranges from 54.15% to 55.27%, and the F1-Macro ranges from 40.30% to 50.36% [1], [2], [5]. The HHEM-2.1-Open variant shows a BA of 51.98% to 54.36% and an F1-Macro of 33.03% to 50.78% [1], [2]. The original HHEM-1 model has a BA of 48.70% to 49.96% and an F1-Macro of 42.37% to 49.02% [1], [2]. These scores indicate the models' performance in detecting hallucinations, with higher scores suggesting better accuracy.

=== Factual Consistency Score: 0.9296875 ===

=== Top Search Results ===

--- Result 1 (score: 0.9618) ---
Document: hallucination-detection-naacl.pdf
Title: Hallucination Detection in RAG Systems
Text: Hallucination Detector:HHEM-2.1 (Men

In [7]:
# Compare results: score changes and document reordering
if no_glossary_result and with_glossary_result:
    print("=== Abbreviation Resolution Comparison ===")
    print(f"\nQuery: \"{ABBREV_QUERY}\"")
    print(f"\nWithout glossary — result count: {len(no_glossary_result.get('search_results', []))}")
    print(f"With glossary    — result count: {len(with_glossary_result.get('search_results', []))}")

    print("\nWithout glossary — top-5 docs:")
    for i, sr in enumerate(no_glossary_result.get('search_results', [])[:5], 1):
        meta = sr.get('document_metadata', {})
        print(f"  {i}. {sr['document_id']} (score: {sr.get('score', 0):.4f}) — {meta.get('title', 'N/A')}")

    print("\nWith glossary — top-5 docs:")
    for i, sr in enumerate(with_glossary_result.get('search_results', [])[:5], 1):
        meta = sr.get('document_metadata', {})
        print(f"  {i}. {sr['document_id']} (score: {sr.get('score', 0):.4f}) — {meta.get('title', 'N/A')}")

    # Check for hallucination-related content surfacing with glossary
    def has_hallucination_content(sr):
        doc_id = sr.get('document_id', '').lower()
        title = sr.get('document_metadata', {}).get('title', '').lower()
        text = sr.get('text', '').lower()
        keywords = ['hallucination', 'factual', 'consistency', 'hhem', 'grounding']
        return any(kw in doc_id or kw in title or kw in text for kw in keywords)

    no_gloss_relevant = sum(1 for sr in no_glossary_result.get('search_results', [])[:5]
                           if has_hallucination_content(sr))
    with_gloss_relevant = sum(1 for sr in with_glossary_result.get('search_results', [])[:5]
                             if has_hallucination_content(sr))

    print(f"\nHallucination-related results in top-5:")
    print(f"  Without glossary: {no_gloss_relevant}")
    print(f"  With glossary:    {with_gloss_relevant}")
    print("\n-> The glossary helps the reranker connect 'HHEM' to hallucination evaluation content.")

=== Abbreviation Resolution Comparison ===

Query: "HHEM accuracy and benchmarks"

Without glossary — result count: 30
With glossary    — result count: 52

Without glossary — top-5 docs:
  1. docs-vectara-com-docs-hallucination-and-evaluation-hallucination-evaluation (score: 0.9878) — Hallucination evaluation
  2. hallucination-detection-naacl.pdf (score: 0.9010) — Hallucination Detection in RAG Systems
  3. hallucination-detection-naacl.pdf (score: 0.8863) — Hallucination Detection in RAG Systems
  4. docs-vectara-com-docs-hallucination-and-evaluation-hallucination-evaluation (score: 0.8856) — Hallucination evaluation
  5. hallucination-detection-naacl.pdf (score: 0.8752) — Hallucination Detection in RAG Systems

With glossary — top-5 docs:
  1. hallucination-detection-naacl.pdf (score: 0.9618) — Hallucination Detection in RAG Systems
  2. hallucination-detection-naacl.pdf (score: 0.8481) — Hallucination Detection in RAG Systems
  3. docs-vectara-com-docs-hallucination-and-evaluation-